## 02.1 — Songs Data Preprocessing


In [ ]:
import os
import math
import glob
import shutil
import h5py
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, lower, trim
from pyspark.sql.functions import when, row_number
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.ml.functions import vector_to_array

## THE NEXT TWO CELLS ARE CRUCIAL FOR RUNNING HADOOP AND PYSPARK ON MY DEVICE


In [3]:
import sys

# On Windows, the system-level SPARK_HOME may point to a different Spark version.
# We override it here to make sure PySpark always finds its own bundled JARs.
os.environ["SPARK_HOME"]  = os.path.dirname(pyspark.__file__)

# Spark needs Hadoop's winutils.exe on Windows to handle local file operations
# (directory creation, temp files, etc.). We installed winutils at C:\hadoop\bin\.
os.environ["HADOOP_HOME"] = r"C:\hadoop"

# Prepend C:\hadoop\bin to PATH so the JVM can find hadoop.dll via
# System.loadLibrary("hadoop"). HADOOP_HOME alone only locates winutils.exe
# for Hadoop's shell helpers — it does NOT add the directory to
# java.library.path. Without this, NativeIO$Windows.access0 throws
# UnsatisfiedLinkError on Spark write operations.
hadoop_bin = r"C:\hadoop\bin"
if hadoop_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = hadoop_bin + os.pathsep + os.environ.get("PATH", "")

# Pin the Python interpreter for both the driver and the executor-side workers
# to the exact Python running this notebook. Without these, PySpark on Windows
# can pick a different Python for workers, causing them to die with
# "Connection reset" the moment Spark tries to deserialize Python rows.
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("SPARK_HOME    :", os.environ["SPARK_HOME"])
print("HADOOP_HOME   :", os.environ["HADOOP_HOME"])
print("PATH[0]       :", os.environ["PATH"].split(os.pathsep)[0])
print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])

SPARK_HOME    : c:\Users\Mariam\anaconda3\envs\spotify-recommender\Lib\site-packages\pyspark
HADOOP_HOME   : C:\hadoop
PATH[0]       : C:\hadoop\bin
PYSPARK_PYTHON: c:\Users\Mariam\anaconda3\envs\spotify-recommender\python.exe


In [4]:
# local[*]  — uses all CPU cores on this machine (pseudo-distributed mode).
# spark.driver.memory — heap given to the JVM driver process.
# spark.local.dir     — MUST be on the same drive as the output directory (D:).
#   Spark writes temp files here, then renames them to the final output path.
#   Java's File.renameTo() cannot rename across different drives on Windows,
#   so if temp is on C: and output is on D:, every write fails silently.
# spark.hadoop.home.dir — tells the JVM where winutils lives at runtime.
# RawLocalFileSystem  — skips Unix chmod/chown calls that crash on Windows.
# fileoutputcommitter.algorithm.version=2 — avoids the rename-based commit phase.
# marksuccessfuljobs=false — skips writing the _SUCCESS marker file.

SPARK_TEMP_DIR = "D:/tmp/spark"
os.makedirs(SPARK_TEMP_DIR, exist_ok=True)

spark = SparkSession.builder \
    .appName("SpotifyPreprocessing") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.local.dir", SPARK_TEMP_DIR) \
    .config("spark.hadoop.home.dir", r"C:\hadoop") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .config("spark.hadoop.fs.file.impl.disable.cache", "true") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.marksuccessfuljobs", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)


def spark_write_csv(df, relative_path):
    """
    Write a Spark DataFrame to a single CSV file on Windows.
    Deletes the output directory with Python first so Spark never has to
    rename or overwrite an existing folder across drives.
    """
    if os.path.exists(relative_path):
        shutil.rmtree(relative_path)
    abs_path = os.path.abspath(relative_path).replace("\\", "/")
    df.coalesce(1).write.csv(f"file:///{abs_path}", header=True)
    print(f"Saved → {relative_path}/")

Spark version: 3.5.8


In [5]:
# The 8 audio features we use across all Spotify datasets.
AUDIO_FEATURES = [
    "danceability", "energy", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]

# Columns we keep from each dataset alongside the audio features.
METADATA_COLS = ["track_id", "track_name", "artist_name", "duration_ms"]

# Output directories
PROCESSED_DIR = "../data/processed"
SPLIT_DIR     = "../outputs/train_test_split"

In [6]:
TARGET_SCHEMA = [
    ("track_id",    "string"),
    ("track_name",  "string"),
    ("artist_name", "string"),
    ("duration_ms", "double"),
    *[(f, "double") for f in AUDIO_FEATURES],
]


def to_target(df, mapping):
    cols = []
    for name, dtype in TARGET_SCHEMA:
        if name in mapping:
            cols.append(col(mapping[name]).cast(dtype).alias(name))
        else:
            cols.append(lit(None).cast(dtype).alias(name))
    return df.select(*cols)


In [7]:
print("Reading raw Spotify 2023 datasets...")
df2_albums = spark.read.csv(
    "../data/raw/spotify_2023/spotify-albums_data_2023.csv",
    header=True, inferSchema=True,
)
df2_features = spark.read.csv(
    "../data/raw/spotify_2023/spotify_features_data_2023.csv",
    header=True, inferSchema=True,
)

df2 = (
    df2_albums.select("track_id", "artist_id", "duration_ms", "track_name", "artist_0")
    .join(
        df2_features.select(
            col("id").alias("track_id"),
            "danceability", "loudness", "tempo", "energy",
            "liveness", "speechiness", "acousticness",
            "instrumentalness", "valence",
        ),
        on="track_id",
        how="left",
    )
)
print("Raw Spotify 2023 schema and sample:")
df2.printSchema()
df2.show(5, truncate=False)

Reading raw Spotify 2023 datasets...
Raw Spotify 2023 schema and sample:
root
 |-- track_id: string (nullable = true)
 |-- artist_id: string (nullable = true)
 |-- duration_ms: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_0: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- valence: double (nullable = true)

+----------------------+----------------------+-----------+---------------------------------------------------------------------------------------------------------+--------------------------+------------+--------+-------+------+--------+-----------+------------+----------------+-------+
|track_id              |artist_id             |durati

In [8]:
# ---- Dataset 1: dataset_of_songs/genres_v2.csv ----
print("Dataset 1: dataset_of_songs/genres_v2.csv")
df1_raw = spark.read.csv(
    "../data/raw/dataset_of_songs/genres_v2.csv",
    header=True, inferSchema=True,
)
df1_std = to_target(df1_raw, {
    "track_id":         "id",
    "track_name":       "song_name",
    "duration_ms":      "duration_ms",
    "danceability":     "danceability",
    "energy":           "energy",
    "tempo":            "tempo",
    "liveness":         "liveness",
    "valence":          "valence",
    "speechiness":      "speechiness",
    "acousticness":     "acousticness",
    "instrumentalness": "instrumentalness",
})
print(f"Dataset 1 row count: {df1_std.count():,}")
df1_std.printSchema()
df1_std.show(5, truncate=False)
# ---- Dataset 2: spotify_2023 (df2 already built above) ----
print("Dataset 2: spotify_2023 (built from spotify-albums_data_2023.csv + spotify_features_data_2023.csv)")
df2_std = to_target(df2, {
    "track_id":         "track_id",
    "track_name":       "track_name",
    "artist_name":      "artist_0",
    "duration_ms":      "duration_ms",
    "danceability":     "danceability",
    "energy":           "energy",
    "tempo":            "tempo",
    "liveness":         "liveness",
    "valence":          "valence",
    "speechiness":      "speechiness",
    "acousticness":     "acousticness",
    "instrumentalness": "instrumentalness",
})
print(f"Dataset 2 row count: {df2_std.count():,}")
df2_std.printSchema()
df2_std.show(5, truncate=False)
# ---- Dataset 3: spotify_audio_features (April 2019 + Nov 2018, same schema) ----
print("Dataset 3: spotify_audio_features (April 2019 + Nov 2018, same schema)")
df3_raw = (
    spark.read.csv(
        "../data/raw/spotify_audio_features/SpotifyAudioFeaturesApril2019.csv",
        header=True, inferSchema=True)
    .unionByName(spark.read.csv(
        "../data/raw/spotify_audio_features/SpotifyAudioFeaturesNov2018.csv",
        header=True, inferSchema=True))
)
df3_std = to_target(df3_raw, {
    "track_id":         "track_id",
    "track_name":       "track_name",
    "artist_name":      "artist_name",
    "duration_ms":      "duration_ms",
    "danceability":     "danceability",
    "energy":           "energy",
    "tempo":            "tempo",
    "liveness":         "liveness",
    "valence":          "valence",
    "speechiness":      "speechiness",
    "acousticness":     "acousticness",
    "instrumentalness": "instrumentalness",
})
print(f"Dataset 3 row count: {df3_std.count():,}")
df3_std.printSchema()
df3_std.show(5, truncate=False)

# ---- Dataset 4: spotify_tracks/dataset.csv ----
print("Dataset 4: spotify_tracks/dataset.csv")
df4_raw = spark.read.csv(
    "../data/raw/spotify_tracks/dataset.csv",
    header=True, inferSchema=True,
)
df4_std = to_target(df4_raw, {
    "track_id":         "track_id",
    "artist_name":      "artists",
    "track_name":       "track_name",
    "duration_ms":      "duration_ms",
    "danceability":     "danceability",
    "energy":           "energy",
    "tempo":            "tempo",
    "liveness":         "liveness",
    "valence":          "valence",
    "speechiness":      "speechiness",
    "acousticness":     "acousticness",
    "instrumentalness": "instrumentalness",
})
print(f"Dataset 4 row count: {df4_std.count():,}")
df4_std.printSchema()
df4_std.show(5, truncate=False)
# ---- Dataset 5: ultimate_spotify_tracks/SpotifyFeatures.csv ----
print("Dataset 5: ultimate_spotify_tracks/SpotifyFeatures.csv")
df5_raw = spark.read.csv(
    "../data/raw/ultimate_spotify_tracks/SpotifyFeatures.csv",
    header=True, inferSchema=True,
)
df5_std = to_target(df5_raw, {
    "track_id":         "track_id",
    "artist_name":      "artist_name",
    "track_name":       "track_name",
    "duration_ms":      "duration_ms",
    "danceability":     "danceability",
    "energy":           "energy",
    "tempo":            "tempo",
    "liveness":         "liveness",
    "valence":          "valence",
    "speechiness":      "speechiness",
    "acousticness":     "acousticness",
    "instrumentalness": "instrumentalness",
})
print(f"Dataset 5 row count: {df5_std.count():,}")
df5_std.printSchema()
df5_std.show(5, truncate=False)





Dataset 1: dataset_of_songs/genres_v2.csv
Dataset 1 row count: 42,305
root
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- duration_ms: double (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)

+----------------------+---------------------------------------------+-----------+-----------+-------------------+------------------+-------------------+------------+----------------+-------------------+-------+------------------+
|track_id              |track_name                                   |artist_name|duration_ms|danceability       |energy            |speechiness        |acousticness|instrumentalness|liveness 

In [10]:
# Prefer rows that have artist_name populated; break ties by total non-null fields.
score_cols = [c for c, _ in TARGET_SCHEMA if c != "track_id"]
nonnull_score = sum(when(col(c).isNotNull(), 1).otherwise(0) for c in score_cols)
has_artist   = when(col("artist_name").isNotNull() & (col("artist_name") != ""), 1).otherwise(0)

w = Window.partitionBy("track_id").orderBy(
    has_artist.desc(),          # rows with an artist_name come first
    col("_score").desc(),       # then the most complete row wins
)
songs_df_merged = (
    df1_std.unionByName(df2_std)
           .unionByName(df3_std)
           .unionByName(df4_std)
           .unionByName(df5_std)
           .withColumn("_score", nonnull_score)
           .withColumn("row_num", row_number().over(w))
           .filter(col("row_num") == 1)
           .drop("_score", "row_num")
)

print(f"Merged dataset row count: {songs_df_merged.count():,}")
print("Merged dataset schema and sample:")
songs_df_merged.printSchema()
songs_df_merged.show(5, truncate=False)

Merged dataset row count: 830,446
Merged dataset schema and sample:
root
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- duration_ms: double (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)

+--------------------------------------------------+-------------------------------------------------------------+----------------------+-----------+------------+------+-----------+------------+----------------+--------+-------+-----+
|track_id                                          |track_name                                                   |artist_name           |duration_ms|danceability|energy|speechiness|acousticn

In [11]:
# Cache the merged DataFrame so Spark doesn't recompute it from scratch
# every time we call count() or run an action below.
songs_df_merged.cache()

# Step 1: Drop rows where any audio feature is null.
# We allow null metadata (genre, artist) — those won't break the ML models.
print("running step 1: dropping rows with null audio features...")
songs_df_cleaned = songs_df_merged.dropna(subset=AUDIO_FEATURES)

# Step 2: Remove duplicate track IDs — same song appearing in multiple datasets.
print("running step 2: dropping duplicate track IDs...")
songs_df_cleaned = songs_df_cleaned.dropDuplicates(["track_id"])

# Step 3: Remove songs with the same name + artist but different IDs
# (e.g., a track re-uploaded to Spotify gets a new track_id but is the same song).
print("running step 3: dropping songs with same name + artist but different IDs...")
songs_df_cleaned = songs_df_cleaned.dropDuplicates(["track_name", "artist_name"])


#step 4: set any null artist names to "Unknown Artist" so we don't lose those songs in the final dataset
print("running step 4: setting null/empty artist names to 'Unknown Artist'...")
songs_df_cleaned = songs_df_cleaned.withColumn(
    "artist_name",
    when(col("artist_name").isNull() | (col("artist_name") == "") | (col("artist_name") == " ") | (col("artist_name") == None ), "Unknown Artist").otherwise(col("artist_name"))
)


#step 5: trim whitespace and lowercase track and artist names for better matching in the recommender system
print("running step 5: trimming whitespace and lowercasing track and artist names...")
songs_df_cleaned = songs_df_cleaned.withColumn(
    "track_name", lower(trim(col("track_name")))
).withColumn(
    "artist_name", lower(trim(col("artist_name")))
)



#step 6: remove any songs that have duration less than 30 seconds or greater than 15 minutes (900,000 ms) as those are likely to be outliers or non-music tracks that could skew the models
print("running step 6: removing songs with duration less than 30 seconds or greater than 15 minutes...")
songs_df_cleaned = songs_df_cleaned.filter(
    (col("duration_ms") >= 30000) & (col("duration_ms") <= 900000)
)

#step 7: rename duration_ms to duration for better readability and to match the feature name used in the Spotify API
print("running step 7: renaming duration_ms to duration for better readability...")
songs_df_cleaned = songs_df_cleaned.withColumnRenamed("duration_ms", "duration")

#step 8: use min-max normalization to scale all audio features and duration to 0-1 range so the ML models can learn better and we don't have to worry about scale differences across features
print("running step 8: using min-max normalization to scale all audio features and duration to 0-1 range...")
print("sample data before normalization:")
songs_df_cleaned.show(5)
feature_cols = ["duration", "danceability", "energy", "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "tempo"]
# Pack the feature columns into a Vector (MinMaxScaler requires Vector input),
# fit on this dataset's min/max, then unpack the scaled Vector back into
# individual columns so downstream code (CSV writes, similarity) can use them directly.
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_vec")
assembled = assembler.transform(songs_df_cleaned)
scaler_model = MinMaxScaler(inputCol="features_vec", outputCol="features_scaled").fit(assembled)
scaled = scaler_model.transform(assembled).withColumn("_arr", vector_to_array("features_scaled"))
for i, c in enumerate(feature_cols):
    scaled = scaled.withColumn(c, col("_arr")[i])
songs_df_cleaned = scaled.drop("features_vec", "features_scaled", "_arr")
print("sample data after normalization:")
songs_df_cleaned.show(5)

#step 9: trim whitespace and lowercase track and artist names for better matching in the recommender system (again, after all the cleaning steps to catch any new nulls or duplicates that got introduced)
print("running step 9: trimming whitespace and lowercasing track and artist names...")
songs_df_cleaned = songs_df_cleaned.withColumn(
    "track_name", lower(trim(col("track_name")))
).withColumn(
    "artist_name", lower(trim(col("artist_name")))
)






total = songs_df_merged.count()
after = songs_df_cleaned.count()
print(f"Rows removed        : {total - after:,}")
print(f"Final shape: {after:,} rows × {len(songs_df_cleaned.columns)} cols")



os.makedirs(PROCESSED_DIR, exist_ok=True)
spark_write_csv(songs_df_cleaned, f"{PROCESSED_DIR}/songs_cleaned")

running step 1: dropping rows with null audio features...
running step 2: dropping duplicate track IDs...
running step 3: dropping songs with same name + artist but different IDs...
running step 4: setting null/empty artist names to 'Unknown Artist'...
running step 5: trimming whitespace and lowercasing track and artist names...
running step 6: removing songs with duration less than 30 seconds or greater than 15 minutes...
running step 7: renaming duration_ms to duration for better readability...
running step 8: using min-max normalization to scale all audio features and duration to 0-1 range...
sample data before normalization:
+--------------------+--------------------+-----------------+--------+------------+------+-----------+------------+----------------+--------+------------------+-------+
|            track_id|          track_name|      artist_name|duration|danceability|energy|speechiness|acousticness|instrumentalness|liveness|           valence|  tempo|
+--------------------+---

In [12]:
# randomSplit is reproducible with seed=42 — same split every run.
os.makedirs(SPLIT_DIR,exist_ok=True)
songs_train_df, songs_test_df = songs_df_cleaned.randomSplit([0.8, 0.2], seed=42)

spark_write_csv(songs_train_df, f"{SPLIT_DIR}/songs_train")
spark_write_csv(songs_test_df,  f"{SPLIT_DIR}/songs_test")

print(f"Train : {songs_train_df.count():,} rows")
print(f"Test  : {songs_test_df.count():,} rows")

Saved → ../outputs/train_test_split/songs_train/
Saved → ../outputs/train_test_split/songs_test/
Train : 591,514 rows
Test  : 147,390 rows


In [ ]:
spark.stop()
print("Done. Spark session closed.")